# Step 04 — Siamese CNN Training with InfoNCE Loss
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

Trains a Siamese CNN encoder using contrastive self-supervised learning.

**Input (from `data/processed/`):**
- `X_July.npy` — shape `(N, 3, 31, 51)`, z-score normalized
- `labels_aligned.csv` — BSISO phase + ENSO category per day

**Pair design:**
- **Positive** — same BSISO phase + same ENSO, different year
- **Hard negative** — same BSISO phase + different ENSO
- **Easy negative** — different BSISO phase

**Architecture:** 3-layer CNN → global avg pool → FC → 64-dim L2-normalized embedding  
**Loss:** InfoNCE (NT-Xent), temperature=0.07  
**Training:** 50 epochs, Adam lr=1e-3, CosineAnnealingLR, Colab T4 GPU (~2.5 hrs)

**Outputs (saved to Drive):**
- `checkpoints/encoder_epoch_N.pth` — checkpoints every 10 epochs
- `checkpoints/encoder_final.pth` — final model weights
- `checkpoints/training_history.json` — loss curves
- `results/training_curves.png`

---
## Before Running
1. Make sure notebook 03 has been run and `data/processed/` contains `X_July.npy` and `labels_aligned.csv`
2. Set runtime to GPU: **Runtime → Change runtime type → T4 GPU**

## Cell 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# ── Approach config ──────────────────────────────────────────────────────────
# 'A'   = raw z-score (X_July.npy)
# 'B'   = yearly-mean removed, then z-score (X_July_B.npy)
# 'lee' = Lee et al. (2013): annual cycle + 120-day running mean removed
APPROACH = 'lee'
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR  = f'{PROJECT_DIR}/data/processed'
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints'
RESULTS_DIR    = f'{PROJECT_DIR}/results'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# File names derived from APPROACH
if APPROACH == 'A':
    X_FILE = 'X_July.npy'
elif APPROACH == 'lee':
    X_FILE = 'X_MJJAS_lee.npy'
else:
    X_FILE = f'X_July_{APPROACH}.npy'
if APPROACH == 'lee':
    LABELS_FILE = 'labels_aligned_mjjas_lee.csv'
else:
    LABELS_FILE = 'labels_aligned.csv'
ENCODER_SUFFIX = '' if APPROACH == 'A' else f'_{APPROACH}'  # '' or '_B' or '_lee'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Approach: {APPROACH}  →  loading {X_FILE}')
print(f'Checkpoints will be saved as: encoder_final{ENCODER_SUFFIX}.pth')

## Cell 2 — Load Data + Train/Val Split + Phase-ENSO Index

In [ ]:
X      = np.load(f'{PROCESSED_DIR}/{X_FILE}')
labels = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])

print(f'Approach: {APPROACH}  →  loaded {X_FILE}')
print(f'X shape:  {X.shape}  (N, channels, lat, lon)')
print(f'Labels:   {len(labels)} rows')
print(f'Columns:  {list(labels.columns)}')
print(f'\nBSISO phase distribution:')
print(labels['bsiso_phase'].value_counts().sort_index())
print(f'\nENSO distribution:')
print(labels['enso_category'].value_counts())

# --- Train/val split (80/20, stratified by BSISO phase) ---
indices = np.arange(len(labels))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42,
    stratify=labels['bsiso_phase'].values
)
np.save(f'{PROCESSED_DIR}/train_indices.npy', train_idx)
np.save(f'{PROCESSED_DIR}/val_indices.npy',   val_idx)
print(f'\nTrain: {len(train_idx)} samples  Val: {len(val_idx)} samples')

# --- Build phase-ENSO index (active BSISO only: amplitude > 1.0) ---
phase_enso_index = defaultdict(list)
for idx, row in labels.iterrows():
    if row['bsiso_amplitude'] > 1.0:
        key = (row['bsiso_phase'], row['enso_category'])
        phase_enso_index[key].append(idx)

active_total = sum(len(v) for v in phase_enso_index.values())
print(f'\nActive BSISO days (amplitude > 1.0): {active_total} / {len(labels)}')
print('\nSample counts per (phase, ENSO):')
for key in sorted(phase_enso_index.keys()):
    n = len(phase_enso_index[key])
    flag = '  << sparse' if n < 20 else ''
    print(f'  {key}: {n}{flag}')

## Cell 3 — PairSampler

Samples three types of pairs for contrastive learning:
- **Positive** — same BSISO phase, same ENSO, different year
- **Hard negative** — same BSISO phase, different ENSO ← key for research question
- **Easy negative** — different BSISO phase

In [ ]:
class PairSampler:
    def __init__(self, labels_df, phase_enso_index):
        self.labels = labels_df
        self.index  = phase_enso_index
        self.enso_categories = labels_df['enso_category'].unique().tolist()

    def sample_positive_pair(self):
        """Same phase + same ENSO, different years."""
        key = self._random_category()
        indices = self.index[key]
        if len(indices) < 2:
            return self.sample_easy_negative_pair()
        idx_A, idx_B = np.random.choice(indices, size=2, replace=False)
        # Try to ensure different years
        year_A = self.labels.loc[idx_A, 'date'].year
        other = [i for i in indices if self.labels.loc[i, 'date'].year != year_A]
        if other:
            idx_B = np.random.choice(other)
        return idx_A, idx_B, 'positive'

    def sample_hard_negative_pair(self):
        """Same phase, different ENSO."""
        phase = np.random.choice(range(1, 9))
        enso_A, enso_B = np.random.choice(self.enso_categories, size=2, replace=False)
        key_A, key_B = (phase, enso_A), (phase, enso_B)
        if not self.index[key_A] or not self.index[key_B]:
            return self.sample_positive_pair()
        idx_A = np.random.choice(self.index[key_A])
        idx_B = np.random.choice(self.index[key_B])
        return idx_A, idx_B, 'hard_negative'

    def sample_easy_negative_pair(self):
        """Different phases."""
        phase_A, phase_B = np.random.choice(range(1, 9), size=2, replace=False)
        enso_A = np.random.choice(self.enso_categories)
        enso_B = np.random.choice(self.enso_categories)
        key_A, key_B = (phase_A, enso_A), (phase_B, enso_B)
        idx_A = (np.random.choice(self.index[key_A]) if self.index[key_A]
                 else self.labels[self.labels['bsiso_phase'] == phase_A].sample(1).index[0])
        idx_B = (np.random.choice(self.index[key_B]) if self.index[key_B]
                 else self.labels[self.labels['bsiso_phase'] == phase_B].sample(1).index[0])
        return idx_A, idx_B, 'easy_negative'

    def sample_batch(self, batch_size=64, positive_ratio=0.3, hard_negative_ratio=0.2):
        n_pos      = int(batch_size * positive_ratio)
        n_hard_neg = int(batch_size * hard_negative_ratio)
        n_easy_neg = batch_size - n_pos - n_hard_neg
        pairs = ([self.sample_positive_pair()      for _ in range(n_pos)] +
                 [self.sample_hard_negative_pair() for _ in range(n_hard_neg)] +
                 [self.sample_easy_negative_pair() for _ in range(n_easy_neg)])
        return pairs

    def _random_category(self):
        valid = [k for k in self.index if len(self.index[k]) > 0]
        return valid[np.random.randint(len(valid))]

# Quick test
sampler = PairSampler(labels, phase_enso_index)
batch   = sampler.sample_batch(batch_size=64)
types   = [p[2] for p in batch]
print(f'Batch composition: positive={types.count("positive")},  '
      f'hard_neg={types.count("hard_negative")},  easy_neg={types.count("easy_negative")}')

## Cell 4 — BSISOPairDataset + DataLoader

In [ ]:
class BSISOPairDataset(Dataset):
    """
    PyTorch Dataset for Siamese contrastive training.
    - Train mode: samples a random pair on every __getitem__ call
    - Val mode:   uses pre-built fixed positive pairs for consistent evaluation
    """
    def __init__(self, X_data, labels_df, phase_enso_index,
                 mode='train', train_indices=None):
        self.X       = X_data
        self.labels  = labels_df
        self.sampler = PairSampler(labels_df, phase_enso_index)
        self.mode    = mode
        self.train_indices = (train_indices if train_indices is not None
                              else np.arange(len(X_data)))
        if mode == 'val':
            self.val_pairs = self._create_val_pairs()

    def _create_val_pairs(self):
        """Fixed positive pairs from val indices for consistent evaluation."""
        pairs = []
        val_labels = self.labels.loc[self.train_indices]
        for phase in range(1, 9):
            for enso in self.sampler.enso_categories:
                group = val_labels[
                    (val_labels['bsiso_phase'] == phase) &
                    (val_labels['enso_category'] == enso)
                ].index.tolist()
                for i in range(len(group)):
                    for j in range(i + 1, len(group)):
                        pairs.append((group[i], group[j], 'positive'))
        return pairs[:1000]  # cap at 1000 for efficiency

    def __len__(self):
        return (len(self.train_indices) if self.mode == 'train'
                else len(self.val_pairs))

    def __getitem__(self, idx):
        if self.mode == 'train':
            r = np.random.rand()
            if r < 0.30:
                idx_A, idx_B, _ = self.sampler.sample_positive_pair()
            elif r < 0.50:
                idx_A, idx_B, _ = self.sampler.sample_hard_negative_pair()
            else:
                idx_A, idx_B, _ = self.sampler.sample_easy_negative_pair()
        else:
            idx_A, idx_B, _ = self.val_pairs[idx]
        field_A = torch.from_numpy(self.X[idx_A]).float()
        field_B = torch.from_numpy(self.X[idx_B]).float()
        return field_A, field_B


# Create datasets
train_dataset = BSISOPairDataset(X, labels, phase_enso_index,
                                  mode='train', train_indices=train_idx)
val_dataset   = BSISOPairDataset(X, labels, phase_enso_index,
                                  mode='val',   train_indices=val_idx)

train_loader = DataLoader(train_dataset, batch_size=64,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train dataset: {len(train_dataset)} items  →  {len(train_loader)} batches/epoch')
print(f'Val dataset:   {len(val_dataset)} pairs   →  {len(val_loader)} batches')

# Test one batch
batch_A, batch_B = next(iter(train_loader))
print(f'\nBatch A shape: {batch_A.shape}  (should be [64, 3, 31, 51])')
print(f'Batch B shape: {batch_B.shape}')

## Cell 5 — CNN Encoder

3-block CNN: Conv+BN+ReLU+MaxPool → global avg pool → FC → L2-normalize  
Input: `(batch, 3, 31, 51)` → Output: `(batch, 64)` unit-norm embeddings  
~250K parameters

In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self, embedding_dim=64):
        super().__init__()
        # Block 1: 3 → 32 channels, MaxPool /2
        self.conv1 = nn.Conv2d(3,  32, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)          # (32, 15, 25)
        # Block 2: 32 → 64 channels, MaxPool /2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)          # (64, 7, 12)
        # Block 3: 64 → 128 channels (no pooling)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(128)
        # Global pooling + FC
        self.global_pool = nn.AdaptiveAvgPool2d(1)  # (128, 1, 1) — size-agnostic
        self.fc          = nn.Linear(128, embedding_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias,   0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))  # (B, 32, 15, 25)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))  # (B, 64,  7, 12)
        x = F.relu(self.bn3(self.conv3(x)))              # (B, 128, 7, 12)
        x = self.global_pool(x).view(x.size(0), -1)     # (B, 128)
        z = self.fc(x)                                   # (B, 64)
        z = F.normalize(z, p=2, dim=1)                  # L2 normalize
        return z

# Test
encoder = CNNEncoder(embedding_dim=64)
total_params = sum(p.numel() for p in encoder.parameters())
print(f'Total parameters: {total_params:,}')

dummy = torch.randn(4, 3, 31, 51)
out   = encoder(dummy)
print(f'Input:  {dummy.shape}')
print(f'Output: {out.shape}   (should be [4, 64])')
print(f'Norms:  {out.norm(dim=1)}  (should all be 1.0)')

## Cell 6 — InfoNCE Loss

NT-Xent (Normalized Temperature-scaled Cross Entropy).  
Pairs are in batch as `(z_A[i], z_B[i])` — positives on the diagonal.  
Expected loss for random embeddings: `log(batch_size)` ≈ `log(64)` ≈ 4.16

In [ ]:
def InfoNCE_loss(z_A, z_B, temperature=0.07):
    """
    Args:
        z_A, z_B: (batch, embedding_dim) — L2-normalized embeddings
        temperature: scalar, controls sharpness
    Returns:
        scalar loss tensor
    """
    # Similarity matrix: sim[i,j] = z_A[i] · z_B[j] / temperature
    sim_matrix = torch.matmul(z_A, z_B.T) / temperature  # (B, B)
    # Positive pairs are on the diagonal: label[i] = i
    labels = torch.arange(z_A.size(0), device=z_A.device)
    return F.cross_entropy(sim_matrix, labels)

# Test
z_A = F.normalize(torch.randn(64, 64), p=2, dim=1)
z_B = F.normalize(torch.randn(64, 64), p=2, dim=1)
loss = InfoNCE_loss(z_A, z_B, temperature=0.07)
print(f'Loss (random embeddings): {loss.item():.4f}')
print(f'Expected ~log(64) = {np.log(64):.4f}')

## Cell 7 — CPU Smoke Test (2 epochs, 100 samples)

Verify the full pipeline (data → model → loss → backward) runs without errors.  
**Expected:** loss decreases from ~4.5 to ~3.5 over 2 epochs.  
This runs on CPU and takes ~1 minute.

In [ ]:
# Small dataset: first 100 samples
X_test      = X[:100]
labels_test = labels.iloc[:100].copy().reset_index(drop=True)

# Rebuild index for test data
pei_test = defaultdict(list)
for idx, row in labels_test.iterrows():
    if row['bsiso_amplitude'] > 1.0:
        pei_test[(row['bsiso_phase'], row['enso_category'])].append(idx)

test_dataset = BSISOPairDataset(X_test, labels_test, pei_test, mode='train')
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=True)

enc_test  = CNNEncoder(embedding_dim=64)   # CPU
opt_test  = optim.Adam(enc_test.parameters(), lr=1e-3)

enc_test.train()
for epoch in range(2):
    epoch_loss = 0.0
    for batch_idx, (fA, fB) in enumerate(test_loader):
        z_A = enc_test(fA)
        z_B = enc_test(fB)
        loss = InfoNCE_loss(z_A, z_B)
        opt_test.zero_grad()
        loss.backward()
        opt_test.step()
        epoch_loss += loss.item()
        if batch_idx % 5 == 0:
            print(f'  Epoch {epoch}  batch {batch_idx}  loss={loss.item():.4f}')
    print(f'Epoch {epoch} avg loss: {epoch_loss/len(test_loader):.4f}\n')

print('CPU smoke test passed!')

## Cell 8 — Initialize Model + Optimizer for Full Training

In [ ]:
# ============================================================
# Training hyperparameters
# ============================================================
EPOCHS      = 50
TEMPERATURE = 0.07
BATCH_SIZE  = 64
LR          = 1e-3
# ============================================================

encoder   = CNNEncoder(embedding_dim=64).to(device)
optimizer = optim.Adam(encoder.parameters(), lr=LR, weight_decay=0)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

total_params = sum(p.numel() for p in encoder.parameters())
print(f'Model on: {device}')
print(f'Parameters: {total_params:,}')
print(f'\nTraining config:')
print(f'  Epochs:      {EPOCHS}')
print(f'  Batch size:  {BATCH_SIZE}')
print(f'  Temperature: {TEMPERATURE}')
print(f'  LR:          {LR}  (cosine decay to 1e-5)')
print(f'  Train batches/epoch: {len(train_loader)}')

## Cell 9 — Full Training Loop

**Expected time:** ~2–3 min/epoch on T4 → ~2.5 hrs for 50 epochs  
Checkpoints saved every 10 epochs to Google Drive.

In [ ]:
from tqdm.notebook import tqdm

history = {'train_loss': [], 'val_loss': [], 'epoch_time': []}

for epoch in range(EPOCHS):
    t0 = time.time()

    # ---- TRAIN ----
    encoder.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False)
    for field_A, field_B in pbar:
        field_A = field_A.to(device, non_blocking=True)
        field_B = field_B.to(device, non_blocking=True)
        z_A = encoder(field_A)
        z_B = encoder(field_B)
        loss = InfoNCE_loss(z_A, z_B, temperature=TEMPERATURE)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    train_loss /= len(train_loader)

    # ---- VALIDATE ----
    encoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for field_A, field_B in val_loader:
            field_A = field_A.to(device, non_blocking=True)
            field_B = field_B.to(device, non_blocking=True)
            z_A = encoder(field_A)
            z_B = encoder(field_B)
            val_loss += InfoNCE_loss(z_A, z_B, temperature=TEMPERATURE).item()
    val_loss /= len(val_loader)

    scheduler.step()

    epoch_time = time.time() - t0
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['epoch_time'].append(epoch_time)

    print(f'Epoch {epoch+1:2d}/{EPOCHS}  '
          f'train={train_loss:.4f}  val={val_loss:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  '
          f'time={epoch_time:.1f}s')

    # Save checkpoint every 10 epochs (named with APPROACH suffix)
    if (epoch + 1) % 10 == 0:
        ckpt_path = f'{CHECKPOINT_DIR}/encoder_epoch_{epoch+1}{ENCODER_SUFFIX}.pth'
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': encoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, ckpt_path)
        print(f'  -> Checkpoint saved: {ckpt_path}')

# Save final model + history (named with APPROACH suffix)
torch.save(encoder.state_dict(), f'{CHECKPOINT_DIR}/encoder_final{ENCODER_SUFFIX}.pth')
with open(f'{CHECKPOINT_DIR}/training_history{ENCODER_SUFFIX}.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete!  [Approach {APPROACH}]')
print(f'Total time: {sum(history["epoch_time"])/60:.1f} minutes')
print(f'Best val loss: {min(history["val_loss"]):.4f}  (epoch {history["val_loss"].index(min(history["val_loss"]))+1})')
print(f'Saved: encoder_final{ENCODER_SUFFIX}.pth  |  training_history{ENCODER_SUFFIX}.json')

## Cell 10 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'],   label='Val Loss',   linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('InfoNCE Loss', fontsize=12)
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)
axes[0].axhline(y=np.log(64), color='gray', linestyle='--', alpha=0.5, label='Random baseline')

# Epoch time
axes[1].plot(history['epoch_time'], color='green', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Time (seconds)', fontsize=12)
axes[1].set_title('Training Time per Epoch', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = f'{RESULTS_DIR}/training_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Avg epoch time: {np.mean(history["epoch_time"]):.1f}s')
print(f'Total time:     {sum(history["epoch_time"])/60:.1f} min')
print(f'Plot saved:     {plot_path}')

## Cell 11 — Post-Training Health Checks

In [ ]:
from scipy.spatial.distance import pdist

print('=' * 55)
print('POST-TRAINING HEALTH CHECKS')
print('=' * 55)

# Check 1: Loss decreased?
loss_drop = (history['train_loss'][0] - history['train_loss'][-1]) / history['train_loss'][0]
if history['train_loss'][-1] < history['train_loss'][0] * 0.9:
    print(f'  Loss decreased {loss_drop*100:.0f}%  ({history["train_loss"][0]:.4f} -> {history["train_loss"][-1]:.4f})')
else:
    print('  WARNING: Loss did not decrease significantly!')
    print('    Possible causes: LR too high/low, bad pair construction, data issue')

# Check 2: Overfitting?
if history['val_loss'][-1] > history['train_loss'][-1] * 1.5:
    print('  WARNING: Possible overfitting (val >> train loss)')
    print('    Consider: add weight_decay, reduce model size, early stopping')
else:
    print(f'  Train/val gap OK  (train={history["train_loss"][-1]:.4f}, val={history["val_loss"][-1]:.4f})')

# Check 3: Embedding collapse?
encoder.eval()
with torch.no_grad():
    sample = torch.from_numpy(X[:100]).float().to(device)
    embeddings_sample = encoder(sample).cpu().numpy()

distances = pdist(embeddings_sample, metric='cosine')
print(f'\nEmbedding diversity (cosine distance on 100 samples):')
print(f'  Mean pairwise distance: {distances.mean():.4f}')
print(f'  Std  pairwise distance: {distances.std():.4f}')
if distances.mean() < 0.1:
    print('  WARNING: Embeddings may have collapsed (all similar)!')
else:
    print('  Embeddings are diverse')

print()
print('Files on Google Drive:')
for fname in ['encoder_final.pth', 'training_history.json',
              'encoder_epoch_10.pth', 'encoder_epoch_20.pth',
              'encoder_epoch_30.pth', 'encoder_epoch_40.pth',
              'encoder_epoch_50.pth']:
    path = f'{CHECKPOINT_DIR}/{fname}'
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1e6
        print(f'  {fname}  ({mb:.2f} MB)')
print('=' * 55)

---
## Done!

If training completed successfully, your Google Drive should have:

```
BSISO_SSL_Project/
├── checkpoints/
│   ├── encoder_epoch_10.pth  ← checkpoint every 10 epochs
│   ├── encoder_epoch_20.pth
│   ├── encoder_epoch_30.pth
│   ├── encoder_epoch_40.pth
│   ├── encoder_epoch_50.pth
│   ├── encoder_final.pth     ← final model weights
│   └── training_history.json ← loss curves
└── results/
    └── training_curves.png
```

**Next step:** Run notebook `05_analysis.ipynb` to extract embeddings, compute t-SNE, and evaluate whether the model learned ENSO-modulated BSISO structure.

---
*DDCS Project | jh9141@nyu.edu*